In [ ]:
%pip install langchain_openrouter

In [ ]:
from langchain_openrouter import ChatOpenRouter
from google.colab import userdata

print(userdata.get("OPENROUTER_API_KEY"))

In [ ]:
# %% Modellinstanz erstellen
MODEL="anthropic/claude-sonnet-4.6"
model=ChatOpenRouter(model=MODEL, api_key=userdata.get("OPENROUTER_API_KEY"))



In [ ]:
query="Was ist Rheinwerk?"
res = model.invoke(query)
# %%
res.model_dump()

**Structured Outputs**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel

Output Parser

In [ ]:
from langchain_core import output_parsers
class MovieOutput(BaseModel):
  title: str
  director: str
  main_actors: list[str]
  release_year: int

class MoviesOutput(BaseModel):
  movies: list[MovieOutput]

output_parser = JsonOutputParser(pydantic_object=MoviesOutput)

output_parser.get_format_instructions()

Prompt Template

In [ ]:
messages = [
    ("system", "Du bist Filmexperte und der Nutzer beschreibt die Handlung. Gib die Top 3 nach kommerziellem Erfolg zurück. Halte dich konkret an die Formatanweisungen {format_instructions}"),
    ("user", "Handlung: {handlung}")
]

prompt_template = ChatPromptTemplate.from_messages(messages=messages).partial(format_instructions=output_parser.get_format_instructions())
prompt_template


Model

In [ ]:
MODEL="anthropic/claude-sonnet-4.6"
model=ChatOpenRouter(model=MODEL, api_key=userdata.get("OPENROUTER_API_KEY"))

Erstellung der Chain

In [ ]:
chain = prompt_template | model | output_parser

Testen der Chain

In [ ]:
user_query = "ein Schiff sinkt"
chain.invoke(input={'handlung': user_query})